# Nested Booking Limits — Interactive Dashboard
Run either collapsed dashboard cell using its play button; Python code is hidden by default.


In [ ]:
#@title Nested Booking Limits — Interactive Dashboard { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

FARES = [('Promo',79), ('Saver',120), ('Standard',190), ('Flex',320)]
CAPACITY = 180
INITIAL_SOLD = [20,20,20,20]
INITIAL_LIMITS = [60,100,140,180]
# Identical illustrative future requests for both policies: low fares arrive first.
REQUESTS = [55,35,30,40]
sold = INITIAL_SOLD.copy()

css = widgets.HTML("""<style>
.rm{max-width:1050px;font-family:Arial,sans-serif;color:#14364a}
.rm h2{margin:5px 0}.rm h3{margin:16px 0 8px}
.rm small,.rm-note{color:#64748b;font-size:12px}
.rm-stats{display:flex;gap:10px;flex-wrap:wrap;margin:14px 0}
.rm-stat{flex:1;min-width:140px;background:#eef3f8;border-radius:10px;padding:13px}
.rm-stat b{display:block;font-size:25px;margin:3px 0}
.rm table{border-collapse:collapse;width:100%;font-size:14px;margin:12px 0}
.rm th{background:#e8f1f7;text-align:left;padding:9px}
.rm td{padding:9px;border-bottom:1px solid #dbe2e8}
.rm-good{color:#087e4e;font-weight:bold}.rm-bad{color:#b91c1c;font-weight:bold}
.rm-track{height:12px;background:#e5e7eb;border-radius:8px;overflow:hidden}
.rm-fill{height:100%;background:#08739a}
</style>""")
output = widgets.Output()
controls = widgets.Output()
buttons = []
sliders = [
    widgets.IntSlider(value=v,min=0,max=180,step=5,description=label,
        style={'description_width':'125px'},layout=widgets.Layout(width='430px'),
        continuous_update=False)
    for v,label in [(60,'Promo limit'),(100,'Saver limit'),(140,'Standard limit')]
]

def limits():
    return [w.value for w in sliders]+[CAPACITY]

def revenue(counts):
    return sum(n*fare for n,(_,fare) in zip(counts,FARES))

def available(counts, caps):
    left=CAPACITY-sum(counts)
    return [max(0,min(left,caps[i]-sum(counts[:i+1]))) for i in range(4)]

def forecast(nested):
    """Apply protection to future requests, never undo existing sales."""
    counts=sold.copy()
    accepted=[0]*4
    rejected=[0]*4
    caps=limits() if nested else [CAPACITY]*4
    for i,n in enumerate(REQUESTS):
        for _ in range(n):
            if sum(counts)<CAPACITY and (not nested or sum(counts[:i+1])<caps[i]):
                counts[i]+=1
                accepted[i]+=1
            else:
                rejected[i]+=1
    return counts,accepted,rejected

def render():
    caps=limits()
    avail=available(sold,caps)
    total=sum(sold)
    rows=''.join(
        f'<tr><td><b>{name}</b><br><small>USD {fare}</small></td>'
        f'<td>{sold[i]}</td><td>{caps[i]}</td><td>{avail[i]}</td>'
        f'<td>USD {sold[i]*fare:,}</td>'
        f'<td class="{"rm-good" if avail[i] else "rm-bad"}">'
        f'{"OPEN" if avail[i] else "CLOSED"}</td></tr>'
        for i,(name,fare) in enumerate(FARES)
    )
    protected,pa,pr=forecast(True)
    baseline,ba,br=forecast(False)
    delta=revenue(protected)-revenue(baseline)
    compare_rows=''.join(
        f'<tr><td>{name}</td><td>{pa[i]}</td><td>{ba[i]}</td>'
        f'<td>{pr[i]}</td><td>{br[i]}</td></tr>'
        for i,(name,_) in enumerate(FARES)
    )
    sign='+' if delta>0 else '-' if delta<0 else ''
    html=f"""
    <div class="rm">
      <small>ILLUSTRATIVE REVENUE MANAGEMENT DASHBOARD</small>
      <h2>Nested Booking Limits | Flight Inventory Control</h2>
      <small>180 seats · Economy cabin · D−14 (14 days before departure) · Illustrative fares and demand</small>
      <div class="rm-stats">
        <div class="rm-stat"><small>Seats sold</small><b>{total}</b></div>
        <div class="rm-stat"><small>Remaining seats</small><b>{CAPACITY-total}</b></div>
        <div class="rm-stat"><small>Load factor</small><b>{total/CAPACITY:.1%}</b></div>
        <div class="rm-stat"><small>Ticket revenue to date</small><b>USD {revenue(sold):,}</b></div>
      </div>
      <div class="rm-track"><div class="rm-fill" style="width:{100*total/CAPACITY:.2f}%"></div></div>
      <table><thead><tr><th>Fare class</th><th>Sold</th><th>Nested limit</th>
      <th>Available</th><th>Revenue to date</th><th>Status</th></tr></thead>
      <tbody>{rows}</tbody></table>
      <div class="rm-note">Available counts overlap and cannot be added. Ticket revenue excludes ancillary sales and costs.</div>
      <h3>Revenue comparison: identical future booking requests</h3>
      <small>From D−14 through departure, future requests arrive in order: Promo {REQUESTS[0]},
        Saver {REQUESTS[1]}, Standard {REQUESTS[2]}, Flex {REQUESTS[3]}.
        Only the inventory policy changes.</small>
      <div class="rm-stats">
        <div class="rm-stat"><small>WITH nested protection · final ticket revenue</small>
          <b>USD {revenue(protected):,}</b><small>{sum(protected)} total seats sold</small></div>
        <div class="rm-stat"><small>WITHOUT protection · first come, first served</small>
          <b>USD {revenue(baseline):,}</b><small>{sum(baseline)} total seats sold</small></div>
      </div>
      <div class="{"rm-good" if delta>=0 else "rm-bad"}">
      Revenue difference (protection minus no protection): {sign}USD {abs(delta):,}</div>
      <table><thead><tr><th>Future fare requests</th>
        <th>Accepted with protection</th><th>Accepted without</th>
        <th>Rejected with protection</th><th>Rejected without</th>
        </tr></thead><tbody>{compare_rows}</tbody></table>
      <div class="rm-note">Deterministic classroom scenario, not an optimized or
      statistically expected-revenue forecast. Protection can help or hurt when
      demand changes. Protection applies to future bookings; past sales remain.</div>
    </div>"""
    with output:
        clear_output(wait=True)
        display(HTML(html))
    for i,button in enumerate(buttons):
        button.disabled=avail[i]<=0

def slider_changed(change=None):
    # Cumulative limits must remain nondecreasing.
    a,b,c=sliders
    a.max=b.value
    b.min=a.value
    b.max=c.value
    c.min=b.value
    render()

for slider in sliders:
    slider.observe(slider_changed,names='value')

def book(i):
    if available(sold,limits())[i]>0:
        sold[i]+=1
        render()

for i,(name,fare) in enumerate(FARES):
    button=widgets.Button(description=f'+1 {name} (USD {fare})',
                          layout=widgets.Layout(width='205px'))
    button.on_click(lambda _,i=i:book(i))
    buttons.append(button)

reset=widgets.Button(description='↻ Reset example',button_style='info',
                     layout=widgets.Layout(width='185px'))
def reset_all(_):
    sold[:]=INITIAL_SOLD
    sliders[0].value=0
    sliders[1].value=0
    sliders[2].value=INITIAL_LIMITS[2]
    sliders[1].value=INITIAL_LIMITS[1]
    sliders[0].value=INITIAL_LIMITS[0]
    render()
reset.on_click(reset_all)

display(css,output,
    widgets.HTML('<b>Accept one booking:</b>'),
    widgets.HBox(buttons,layout=widgets.Layout(flex_flow='row wrap',gap='8px')),
    widgets.HTML('<b>Adjust nested booking limits (cumulative):</b>'),
    widgets.VBox(sliders),reset)
slider_changed()


## Activity 2 — How Many Seats Should We Protect?

**Separate scenario:** A 180-seat flight has **160 seats booked** (20 remain). Today, **15 Saver customers** request seats at **$120**. Before departure, additional **Flex customers** may request seats at **$320**.

**Baseline forecast:** There is an **85% probability that some Flex demand occurs**. If demand occurs, the possible request totals are **5, 10, 15, or 20**, with relative likelihoods proportional to **20:30:25:10**, respectively. This is equivalent to the original probabilities of **15%, 20%, 30%, 25%, 10%** for 0, 5, 10, 15, and 20 requests.

Assume all accepted passengers travel. Exclude revenue from the 160 already booked passengers, which is the same for every decision.

**Questions**
1. At the baseline forecast, how many of the 20 remaining seats should the airline **protect for Flex**? How many Saver requests should it accept?
2. Change the **probability that any Flex demand occurs** to **40%**, then **95%**. How does the revenue-maximizing number of protected seats change?
3. Explain why protecting too few **or** too many seats can reduce expected revenue.

Use the **Seats protected** slider and the updating expected-revenue graph to support your decisions. The demand-size mix stays fixed conditional on Flex demand occurring.

*This activity is independent of the main 80-booking dashboard; all values are illustrative.*

In [ ]:
#@title Activity 2 — Seat Protection and Dynamic Revenue Curve { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt

SEATS_LEFT = 20
SAVER_REQUESTS = 15
SAVER_FARE = 120
FLEX_FARE = 320

# Demand size conditional on there being some Flex demand.
DEMAND_SIZES = [5, 10, 15, 20]
CONDITIONAL_WEIGHTS = [20, 30, 25, 10]
CONDITIONAL_PROBS = [w / sum(CONDITIONAL_WEIGHTS) for w in CONDITIONAL_WEIGHTS]

def probabilities(prob_any):
    """One coherent distribution, with P(no demand) = 1 - P(any demand)."""
    return [1.0 - prob_any] + [prob_any * p for p in CONDITIONAL_PROBS]

def decision(seats_protected, prob_any):
    saver_accepted = min(SAVER_REQUESTS, SEATS_LEFT - seats_protected)
    revenue_today = saver_accepted * SAVER_FARE
    remaining_for_flex = SEATS_LEFT - saver_accepted
    demand_outcomes = [0] + DEMAND_SIZES
    revenue_later = sum(p * min(demand, remaining_for_flex) * FLEX_FARE
                        for demand, p in zip(demand_outcomes, probabilities(prob_any)))
    return saver_accepted, revenue_today, revenue_later, revenue_today + revenue_later

protection = widgets.IntSlider(value=15, min=5, max=20, step=1,
    description='Protect for Flex', continuous_update=False,
    style={'description_width':'155px'}, layout=widgets.Layout(width='500px'))
chance = widgets.IntSlider(value=85, min=0, max=100, step=5,
    description='Chance of any Flex demand', continuous_update=False,
    style={'description_width':'190px'}, layout=widgets.Layout(width='535px'))
activity_output = widgets.Output()

reset_activity = widgets.Button(description='Reset both sliders',
    layout=widgets.Layout(width='190px'))
def reset_controls(_):
    protection.value = 15
    chance.value = 85
    draw_activity()
reset_activity.on_click(reset_controls)

def draw_activity(change=None):
    n = int(protection.value)
    prob_any = chance.value / 100.0
    saver_n, today, later, total = decision(n, prob_any)
    candidates = list(range(5, 21))
    expected = [decision(x, prob_any)[3] for x in candidates]
    baseline = decision(5, prob_any)[3]
    demand_probabilities = probabilities(prob_any)

    prob_row = ' · '.join(
        f'{d} requests: {p:.1%}'
        for d, p in zip([0] + DEMAND_SIZES, demand_probabilities)
    )
    with activity_output:
        clear_output(wait=True)
        display(HTML(
            '<div style="font-family:Arial;color:#07324a;max-width:900px">'
            '<h3>Activity 2: Expected Revenue by Seat Protection Level</h3>'
            '<p>180 seats · 160 already booked · Saver USD 120 · Flex USD 320</p>'
            f'<p><b>Chance of any Flex demand: {chance.value}%</b><br>'
            f'<small>Resulting demand distribution: {prob_row}</small></p>'
            '<table style="width:100%;text-align:center;border-collapse:collapse">'
            '<tr style="background:#e8f1f7"><th>Seats protected for Flex</th>'
            '<th>Saver accepted now</th><th>Revenue now</th>'
            '<th>Expected Flex revenue</th><th>Expected total</th></tr>'
            f'<tr><td>{n}</td><td>{saver_n} of 15</td><td>USD {today:,.0f}</td>'
            f'<td>USD {later:,.0f}</td><td><b>USD {total:,.0f}</b></td></tr>'
            '</table>'
            f'<p style="font-size:13px;color:#64748b">Accept-all Saver benchmark '
            f'(protect 5 seats): USD {baseline:,.0f}. '
            f'Change relative to benchmark: USD {total - baseline:+,.0f}.</p>'
            '<p style="font-size:12px;color:#64748b">'
            'Only the chance of demand changes. The relative mix of positive demand sizes '
            'remains fixed. Existing bookings are not changed.</p></div>'
        ))
        fig, ax = plt.subplots(figsize=(9, 3.6))
        ax.plot(candidates, expected, marker='o', markersize=4, linewidth=2,
                label='Expected incremental revenue')
        ax.scatter([n], [total], color='#e77628', s=100, zorder=4,
                   label='Selected protection level')
        ax.axvline(n, color='#e77628', linestyle=':', linewidth=1)
        ax.set_xlabel('Seats protected for future Flex bookings')
        ax.set_ylabel('Expected incremental ticket revenue (USD)')
        ax.set_title(f'Expected revenue versus protected seats (Flex demand chance = {chance.value}%)')
        ax.set_xticks(candidates)
        ax.grid(alpha=0.25)
        ax.legend(loc='lower left', fontsize=9)
        fig.tight_layout()
        display(fig)
        plt.close(fig)

protection.observe(draw_activity, names='value')
chance.observe(draw_activity, names='value')
display(
    widgets.HTML('<b>Choose seats to protect, then change the chance of later Flex demand:</b>'),
    protection, chance, reset_activity, activity_output
)
draw_activity()
